Modelo

In [ ]:
from models. import Model

In [ ]:
import torch
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split

from dataset import avesDataset, SpectrogramAugment, TransformedSubset
from train import train_model


data_dir = "../dataset"
full_dataset = avesDataset(data_dir)

# Pega o número real de classes do dataset
num_classes = len(full_dataset.label_map)
print(f"[INFO] Classes detectadas: {num_classes}")

# Instancia o modelo com o número correto de classes
model = Model(num_classes=num_classes)

# ──────────────────────────────────────────────────────────────────────────────
# Tópico 2: Split estratificado por label (evita data leakage por classe e
# garante que cada fold tenha a mesma proporção de cada espécie).
# ──────────────────────────────────────────────────────────────────────────────
all_indices = list(range(len(full_dataset)))
all_labels  = [full_dataset.samples[i][1] for i in all_indices]

# 80% treino | 20% temp
train_idx, temp_idx, _, temp_labels = train_test_split(
    all_indices, all_labels,
    test_size=0.2,
    stratify=all_labels,
    random_state=42
)

# 50% do temp → val  |  50% do temp → test  (= 10% / 10% do total)
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.5,
    stratify=temp_labels,
    random_state=42
)

print(f"[INFO] Split estratificado — treino: {len(train_idx)} | val: {len(val_idx)} | teste: {len(test_idx)}")

train_subset = Subset(full_dataset, train_idx)
val_subset   = Subset(full_dataset, val_idx)
test_subset  = Subset(full_dataset, test_idx)

# Augmentation apenas no treino
train_dataset = TransformedSubset(train_subset, transform=SpectrogramAugment())
val_dataset   = val_subset
test_dataset  = test_subset

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

train_model(
    model=model,
    num_epochs=50,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    patience=10,
    min_delta=0.0001,
    lr=0.0001
)


[INFO] Classes detectadas: 27
[INFO] Split estratificado — treino: 8635 | val: 1079 | teste: 1080
Usando dispositivo: cuda
[INFO] Detectadas 27 classes dinamicamente.


c:\Users\Miguel\anaconda3\envs\aves\lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


[INFO] Dataset 'dataset_4ad1c1b' registrado nativamente no MLflow com 10794 amostras.
Epoca [1/50] - Loss Treino: 2.8709, Acc Treino: 23.06% | Loss Val: 1.8410, Acc Val: 58.85%
[OK] Melhor modelo atualizado!
Epoca [2/50] - Loss Treino: 1.7447, Acc Treino: 61.34% | Loss Val: 1.3843, Acc Val: 76.00%
[OK] Melhor modelo atualizado!
Epoca [3/50] - Loss Treino: 1.5059, Acc Treino: 70.06% | Loss Val: 1.2957, Acc Val: 78.68%
[OK] Melhor modelo atualizado!
Epoca [4/50] - Loss Treino: 1.3839, Acc Treino: 74.38% | Loss Val: 1.2561, Acc Val: 79.24%
[OK] Melhor modelo atualizado!
Epoca [5/50] - Loss Treino: 1.2944, Acc Treino: 77.46% | Loss Val: 1.2158, Acc Val: 81.00%
[OK] Melhor modelo atualizado!
Epoca [6/50] - Loss Treino: 1.2295, Acc Treino: 79.75% | Loss Val: 1.1915, Acc Val: 81.74%
[OK] Melhor modelo atualizado!
Epoca [7/50] - Loss Treino: 1.1772, Acc Treino: 81.81% | Loss Val: 1.1793, Acc Val: 81.65%
[OK] Melhor modelo atualizado!
Epoca [8/50] - Loss Treino: 1.1428, Acc Treino: 82.34% | Los

2026/06/08 21:30:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


[OK] Relatorio salvo com sucesso: result\CRNN_EfficientNetB0_BiLSTM\report.pdf
[STATS] Salvando dados no historico geral CSV...
[OK] Dados salvos em result\historico_geral_treinos.csv


...

c:\Users\Miguel\anaconda3\envs\aves\lib\site-packages\mlflow\tracking\_model_registry\utils.py:220: FutureWarning: The filesystem model registry backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri)
Successfully registered model 'CRNN_EfficientNetB0_BiLSTM'.
2026/06/08 21:30:41 WARNING mlflow.tracking._model_registry.fluent: Run with id 4b3bd929ed91478c9fbeac3b2b72ad16 has no artifacts at artifact path 'model', registering model based on models:/m-7c284944cba644b1b9c1e3af86fd16ee instead


[REGISTRY] Modelo 'CRNN_EfficientNetB0_BiLSTM' registrado — versão 1
[REGISTRY] Dataset usado: 4ad1c1b (4ad1c1b6dd0d00930acc944481d23767.dir)


Created version '1' of model 'CRNN_EfficientNetB0_BiLSTM'.


(Model(
   (feature_extractor): EfficientNetExtractor(
     (features): Sequential(
       (0): Conv2dNormActivation(
         (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
         (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
         (2): SiLU(inplace=True)
       )
       (1): Sequential(
         (0): MBConv(
           (block): Sequential(
             (0): Conv2dNormActivation(
               (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
               (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
               (2): SiLU(inplace=True)
             )
             (1): SqueezeExcitation(
               (avgpool): AdaptiveAvgPool2d(output_size=1)
               (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
               (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
               (activation): SiLU(in

: 